In [1]:
from google.colab import files
import pandas as pd
import io
import torch
from sklearn.preprocessing import StandardScaler

print("Please upload your 'yield_df_2.csv' file:")
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

print(f"\nSuccessfully loaded {file_name}!")

Please upload your 'yield_df_2.csv' file:


Saving yield_df.csv to yield_df.csv

Successfully loaded yield_df.csv!


In [2]:
# Dynamically find the yield column and rainfall column
target_col = [col for col in df.columns if 'yield' in col.lower()][0]
rain_col = [col for col in df.columns if 'rain' in col.lower()][0]

# Extract them as 2D NumPy arrays for the scaler
x_raw = df[[rain_col]].values
y_raw = df[[target_col]].values

# Scale the numbers down
scaler_x = StandardScaler()
scaler_y = StandardScaler()

x_scaled = scaler_x.fit_transform(x_raw).flatten()
y_scaled = scaler_y.fit_transform(y_raw).flatten()

print("Data extracted and scaled successfully!")

Data extracted and scaled successfully!


In [3]:
# Check for the T4 GPU
dev = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {dev}")

# Convert arrays to PyTorch Tensors and push to device
x = torch.tensor(x_scaled, dtype=torch.float32, device=dev)
y = torch.tensor(y_scaled, dtype=torch.float32, device=dev)

print(f"X tensor is now on: {x.device}")
print(f"Y tensor is now on: {y.device}")

Using device: cuda
X tensor is now on: cuda:0
Y tensor is now on: cuda:0


In [4]:
# Initialize weights and biases, commanding PyTorch to track them
w = torch.zeros(1, device=dev, requires_grad=True)
b = torch.zeros(1, device=dev, requires_grad=True)

# Set learning rate
lr = 0.1

print("Starting PyTorch Autograd training loop...\n")

for epoch in range(200):
    # 1. Forward Pass: Calculate predictions
    predictions = w * x + b

    # 2. Calculate Error (Mean Squared Error)
    loss = ((predictions - y)**2).mean()

    # 3. Autograd: Calculate the complex math instantly
    loss.backward()

    # 4. Update weights without tracking the update step
    with torch.no_grad():
        w -= lr * w.grad
        b -= lr * b.grad

        # 5. Clear gradients for the next loop (CRITICAL step)
        w.grad.zero_()
        b.grad.zero_()

    if epoch % 40 == 0:
        print(f"Epoch {epoch:3d} | Loss: {loss.item():.4f} | w: {w.item():.4f} | b: {b.item():.4f}")

print("\nTraining complete on real dataset using PyTorch Autograd!")

Starting PyTorch Autograd training loop...

Epoch   0 | Loss: 1.0000 | w: 0.0002 | b: -0.0000
Epoch  40 | Loss: 1.0000 | w: 0.0010 | b: 0.0000
Epoch  80 | Loss: 1.0000 | w: 0.0010 | b: 0.0000
Epoch 120 | Loss: 1.0000 | w: 0.0010 | b: 0.0000
Epoch 160 | Loss: 1.0000 | w: 0.0010 | b: 0.0000

Training complete on real dataset using PyTorch Autograd!
